# Trimodal versus single-modality recovery of the LSC compartment — Patient 1 (03H096 / PB2)

**Role in the paper:** Extended Data Fig. 7 / Supplementary Note 2. It
quantifies how much of the leukemia-stem-cell (LSC) structure seen in the
trimodal MultiVI space is recovered by each single-modality embedding.
This is not trajectory or pseudotime analysis.

**What this notebook does**
1. Loads the cleaned trimodal MultiVI object and the three single-modality
   latent spaces (scVI for RNA, PeakVI for ATAC, CytoVI for protein)
2. Defines the LSC label from the trimodal clustering (`Cluster_Final == "6"`)
3. Scores every latent space with four metrics: k-NN LSC purity, neighbourhood
   preservation against the trimodal graph, silhouette and separation ratio
4. Builds a random-label baseline (1,000 permutations).
5. Writes the statistics tables and the figure panels

**Objects**
- **Reads:** `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/
  Teaseq_Multi_VI_PB2_Cleaned.h5ad"` plus `ScVI_PB2.h5ad`, `PeakVI_PB2.h5ad`
  and `PB2_CytoVI_adata.h5ad` from `04_Single_modality_checks`
- **Creates:** statistics tables (`LSC_*.csv` / `.xlsx`) and figures under
  `OUT_DIR`

The single-modality embeddings are produced by the notebooks in
`04_Single_modality_checks/`.

## Paths and settings

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

from scipy.spatial.distance import cdist
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/LSC_recovery"
OUT_DIR.mkdir(parents=True, exist_ok=True)

patient = "PB2"


## Load the trimodal reference

In [ ]:
# Cleaned trimodal MultiVI object: the reference representation.
adata = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad")

## Load the single-modality latent spaces

In [ ]:
# One latent space per modality, produced in 04_Single_modality_checks.
base_dir = DATA_DIR / "04_Single_modality_checks"

paths = {
    "rna": base_dir / "RNA" / f"ScVI_{patient}.h5ad",
    "atac": base_dir / "ATAC" / f"PeakVI_{patient}.h5ad",
    "protein": base_dir / "Protein" / f"{patient}_CytoVI_adata.h5ad",
}

# CHECK FILES
for key, path in paths.items():
    print(f"{key.upper()} exists:", path.exists(), "|", path)

modalities = {}

for key, path in paths.items():
    if path.exists():
        modalities[key] = sc.read_h5ad(path)
    else:
        print(f"Missing file for {key}: {path}")

print("LOAD DONE")

# ACCESS
adata_rna = modalities.get("rna")
adata_atac = modalities.get("atac")
adata_protein = modalities.get("protein")

# QUICK CHECK
for key, obj in modalities.items():
    print(f"{key.upper()} →", obj)

## LSC recovery metrics

In [ ]:
rna = adata_rna.copy()
atac = adata_atac.copy()
protein = adata_protein.copy()

# Analysis parameters (the only values that change between patients).
lsc_obs_col = "Cluster_Final"   # column holding the trimodal clusters
lsc_value   = "6"               # cluster called LSC for this patient
k = 30                          # neighbours used by every metric
random_state = 0
n_permutations = 1000           # random-label permutations

### 1. Match cells across the four representations

In [ ]:
shared_cells = (
    adata.obs_names
    .intersection(rna.obs_names)
    .intersection(atac.obs_names)
    .intersection(protein.obs_names)
)

if len(shared_cells) == 0:
    raise ValueError("No shared cells across adata, rna, atac, and protein.")

adata_sub   = adata[shared_cells].copy()
rna_sub     = rna[shared_cells].copy()
atac_sub    = atac[shared_cells].copy()
protein_sub = protein[shared_cells].copy()

print("Shared cells:", len(shared_cells))

### 2. Define the LSC label from the trimodal clustering

In [ ]:
adata_sub.obs["LSC_truth"] = (
    adata_sub.obs[lsc_obs_col].astype(str) == str(lsc_value)
).astype(int)

rna_sub.obs["LSC_truth"]     = adata_sub.obs["LSC_truth"].values
atac_sub.obs["LSC_truth"]    = adata_sub.obs["LSC_truth"].values
protein_sub.obs["LSC_truth"] = adata_sub.obs["LSC_truth"].values

n_lsc = int(adata_sub.obs["LSC_truth"].sum())
n_non = int((adata_sub.obs["LSC_truth"] == 0).sum())

print("LSC cells:", n_lsc)
print("Non-LSC cells:", n_non)

if n_lsc < 5:
    raise ValueError("Too few LSC cells for robust analysis.")

### 3. Collect the latent matrices

In [ ]:
X_tri     = adata_sub.obsm["X_multivi"]
X_rna     = rna_sub.obsm["X_scVI"]
X_atac    = atac_sub.obsm["X_peakvi"]
X_protein = protein_sub.obsm["X_CytoVI"]

y = adata_sub.obs["LSC_truth"].values.astype(int)

### 4. Metric definitions

Four complementary metrics are used: local purity of the LSC neighbourhood, Jaccard overlap of each cell's neighbours with its trimodal neighbours, the silhouette of LSC versus the rest, and the ratio of between-group to within-group median distances.

In [ ]:
def knn_indices(X, k):
    k_eff = min(k, X.shape[0] - 1)

    if k_eff < 1:
        raise ValueError("Not enough cells for KNN calculation.")

    nn = NearestNeighbors(n_neighbors=k_eff + 1, metric="euclidean")
    nn.fit(X)
    idx = nn.kneighbors(X, return_distance=False)

    return idx[:, 1:]


def lsc_knn_purity(X, y, k=30):
    idx = knn_indices(X, k)
    lsc_cells = np.where(y == 1)[0]

    scores = []

    for i in lsc_cells:
        neigh = idx[i]
        scores.append(np.mean(y[neigh] == 1))

    return float(np.mean(scores)), np.array(scores)


def neighbor_jaccard_vs_reference(X_ref, X_test, y, k=30):
    idx_ref = knn_indices(X_ref, k)
    idx_test = knn_indices(X_test, k)

    lsc_cells = np.where(y == 1)[0]

    jac = []

    for i in lsc_cells:
        a = set(idx_ref[i])
        b = set(idx_test[i])
        jac.append(len(a & b) / len(a | b))

    return float(np.mean(jac)), np.array(jac)


def separation_ratio(X, y, max_cells=1000, random_state=0):
    rng = np.random.default_rng(random_state)

    lsc_idx = np.where(y == 1)[0]
    non_idx = np.where(y == 0)[0]

    if len(lsc_idx) < 2 or len(non_idx) < 2:
        return np.nan, np.nan, np.nan, np.nan

    lsc_sample = lsc_idx if len(lsc_idx) <= max_cells else rng.choice(lsc_idx, max_cells, replace=False)
    non_sample = non_idx if len(non_idx) <= max_cells else rng.choice(non_idx, max_cells, replace=False)

    X_lsc = X[lsc_sample]
    X_non = X[non_sample]

    d_within = cdist(X_lsc, X_lsc, metric="euclidean")
    d_between = cdist(X_lsc, X_non, metric="euclidean")

    d_within = d_within[np.triu_indices_from(d_within, k=1)]

    within_median = float(np.median(d_within))
    between_median = float(np.median(d_between))

    ratio = within_median / between_median
    inverse_ratio = between_median / within_median

    return within_median, between_median, ratio, inverse_ratio


def silhouette_lsc_vs_rest(X, y):
    if len(np.unique(y)) < 2:
        return np.nan

    return float(silhouette_score(X, y, metric="euclidean"))


def evaluate_space(name, X, y, X_ref=None, k=30, random_state=0):
    purity_mean, purity_per_cell = lsc_knn_purity(X, y, k=k)
    sil = silhouette_lsc_vs_rest(X, y)
    within_med, between_med, ratio, inv_ratio = separation_ratio(
        X,
        y,
        random_state=random_state
    )

    out = {
        "space": name,
        "n_cells": X.shape[0],
        "n_dims": X.shape[1],
        "n_lsc": int(y.sum()),
        "knn_lsc_purity_mean": purity_mean,
        "silhouette_lsc_vs_rest": sil,
        "within_lsc_median_dist": within_med,
        "between_lsc_nonlsc_median_dist": between_med,
        "within_between_ratio_lower_is_better": ratio,
        "between_within_ratio_higher_is_better": inv_ratio,
    }

    per_cell = {
        "knn_lsc_purity_per_cell": purity_per_cell,
    }

    if X_ref is not None:
        jacc_mean, jacc_per_cell = neighbor_jaccard_vs_reference(
            X_ref,
            X,
            y,
            k=k
        )

        out["neighbor_jaccard_vs_trimodal_mean"] = jacc_mean
        per_cell["neighbor_jaccard_vs_trimodal_per_cell"] = jacc_per_cell

    else:
        out["neighbor_jaccard_vs_trimodal_mean"] = 1.0
        per_cell["neighbor_jaccard_vs_trimodal_per_cell"] = np.ones(int(y.sum()))

    return out, per_cell

### 5. Observed metrics per latent space

In [ ]:
spaces = {
    "trimodal_X_multivi": X_tri,
    "rna_X_scVI": X_rna,
    "atac_X_peakvi": X_atac,
    "protein": X_protein,
}

results = []
per_cell_results = {}

for name, X in spaces.items():
    out, per_cell = evaluate_space(
        name,
        X,
        y,
        X_ref=X_tri,
        k=k,
        random_state=random_state
    )

    results.append(out)
    per_cell_results[name] = per_cell

results_df = pd.DataFrame(results)

### 6. Express every metric relative to the trimodal reference

In [ ]:
trimodal_row = results_df.loc[
    results_df["space"] == "trimodal_X_multivi"
].iloc[0]

higher_better_cols = [
    "knn_lsc_purity_mean",
    "silhouette_lsc_vs_rest",
    "between_within_ratio_higher_is_better",
    "neighbor_jaccard_vs_trimodal_mean",
]

lower_better_cols = [
    "within_between_ratio_lower_is_better",
]

for col in higher_better_cols:
    results_df[col + "_vs_trimodal"] = results_df[col] / trimodal_row[col]

for col in lower_better_cols:
    results_df[col + "_vs_trimodal"] = trimodal_row[col] / results_df[col]

composite_cols = [
    "knn_lsc_purity_mean_vs_trimodal",
    "silhouette_lsc_vs_rest_vs_trimodal",
    "between_within_ratio_higher_is_better_vs_trimodal",
    "neighbor_jaccard_vs_trimodal_mean_vs_trimodal",
]

results_df["composite_recovery_score"] = results_df[composite_cols].mean(axis=1)

### 7. Random-label baseline

The LSC labels are permuted 1000 times per latent space to obtain the null distribution of each metric.

In [ ]:
metric_cols_for_random = [
    "knn_lsc_purity_mean",
    "silhouette_lsc_vs_rest",
    "between_within_ratio_higher_is_better",
    "neighbor_jaccard_vs_trimodal_mean",
]

rng_perm = np.random.default_rng(random_state)
perm_results = {}

for name, X in spaces.items():
    print(f"Running random-label baseline for {name}...")

    perm_store = {metric: [] for metric in metric_cols_for_random}
    perm_store["composite_recovery_score"] = []

    for perm_i in range(n_permutations):
        y_perm = rng_perm.permutation(y)

        perm_eval, _ = evaluate_space(
            name=name,
            X=X,
            y=y_perm,
            X_ref=X_tri,
            k=k,
            random_state=random_state + perm_i + 1,
        )

        for metric in metric_cols_for_random:
            perm_store[metric].append(perm_eval[metric])

        perm_norm_values = []

        for metric in metric_cols_for_random:
            trimodal_val = trimodal_row[metric]
            test_val = perm_eval[metric]

            if pd.isna(test_val) or pd.isna(trimodal_val) or trimodal_val == 0:
                perm_norm_values.append(np.nan)
            else:
                perm_norm_values.append(test_val / trimodal_val)

        perm_store["composite_recovery_score"].append(
            float(np.nanmean(perm_norm_values))
        )

    perm_results[name] = {
        metric: np.array(vals, dtype=float)
        for metric, vals in perm_store.items()
    }

for metric in metric_cols_for_random:
    baseline_mean_col = metric + "_random_baseline_mean"
    baseline_sd_col = metric + "_random_baseline_sd"
    p_vs_random_col = metric + "_p_vs_random"

    baseline_means = []
    baseline_sds = []
    pvals = []

    for _, row in results_df.iterrows():
        name = row["space"]
        obs = row[metric]
        perm_vals = perm_results[name][metric]
        perm_vals = perm_vals[~np.isnan(perm_vals)]

        baseline_means.append(float(np.mean(perm_vals)))
        baseline_sds.append(float(np.std(perm_vals, ddof=1)))

        pvals.append(
            float((np.sum(perm_vals >= obs) + 1) / (len(perm_vals) + 1))
        )

    results_df[baseline_mean_col] = baseline_means
    results_df[baseline_sd_col] = baseline_sds
    results_df[p_vs_random_col] = pvals


# composite random baseline
composite_random_means = []
composite_random_sds = []
composite_pvals = []

for _, row in results_df.iterrows():
    name = row["space"]

    perm_composite = perm_results[name]["composite_recovery_score"]
    perm_composite = perm_composite[~np.isnan(perm_composite)]

    obs_composite = row["composite_recovery_score"]

    composite_random_means.append(float(np.mean(perm_composite)))
    composite_random_sds.append(float(np.std(perm_composite, ddof=1)))

    composite_pvals.append(
        float((np.sum(perm_composite >= obs_composite) + 1) / (len(perm_composite) + 1))
    )

results_df["composite_random_baseline_mean"] = composite_random_means
results_df["composite_random_baseline_sd"] = composite_random_sds
results_df["composite_p_vs_random"] = composite_pvals

### 9. Full results table

In [ ]:
pd.set_option("display.max_columns", None)

print("\n=== FULL RESULTS ===")
print(results_df.round(4))

### 10. Summary table

In [ ]:
summary_cols = [
    "space",
    "knn_lsc_purity_mean",
    "knn_lsc_purity_mean_random_baseline_mean",
    "knn_lsc_purity_mean_p_vs_random",
    "neighbor_jaccard_vs_trimodal_mean",
    "neighbor_jaccard_vs_trimodal_mean_random_baseline_mean",
    "neighbor_jaccard_vs_trimodal_mean_p_vs_random",
    "silhouette_lsc_vs_rest",
    "silhouette_lsc_vs_rest_random_baseline_mean",
    "silhouette_lsc_vs_rest_p_vs_random",
    "between_within_ratio_higher_is_better",
    "between_within_ratio_higher_is_better_random_baseline_mean",
    "between_within_ratio_higher_is_better_p_vs_random",
    "composite_recovery_score",
    "composite_random_baseline_mean",
    "composite_p_vs_random",
]
summary_cols = [col for col in summary_cols if col in results_df.columns]
summary_df = results_df[summary_cols].copy()
print("\n=== SUMMARY ===")
print(summary_df.round(4))


### 11. Save the statistics tables

In [ ]:
output_dir = OUT_DIR
print("\nSaving outputs to:", output_dir)


full_csv_path = output_dir / "LSC_full_statistics_with_baseline_and_pvalues.csv"
full_xlsx_path = output_dir / "LSC_full_statistics_with_baseline_and_pvalues.xlsx"
summary_csv_path = output_dir / "LSC_recovery_summary.csv"
summary_xlsx_path = output_dir / "LSC_recovery_summary.xlsx"

results_df.to_csv(full_csv_path, index=False)
results_df.to_excel(full_xlsx_path, index=False)

summary_df.to_csv(summary_csv_path, index=False)
summary_df.to_excel(summary_xlsx_path, index=False)

print("Saved full table:", full_csv_path)
print("Saved full table:", full_xlsx_path)
print("Saved summary table:", summary_csv_path)
print("Saved summary table:", summary_xlsx_path)

### 12. Figure-ready table

In [ ]:
plot_df = results_df.copy()

plot_df = plot_df[[
    "space",
    "knn_lsc_purity_mean",
    "neighbor_jaccard_vs_trimodal_mean",
    "silhouette_lsc_vs_rest",
    "between_within_ratio_higher_is_better",
    "composite_recovery_score",
    "composite_random_baseline_mean",
    "composite_p_vs_random",
]]

plot_df = plot_df.rename(columns={
    "space": "Modality",
    "knn_lsc_purity_mean": "LSC Local Purity",
    "neighbor_jaccard_vs_trimodal_mean": "Neighborhood Preservation",
    "silhouette_lsc_vs_rest": "LSC Separation (Silhouette)",
    "between_within_ratio_higher_is_better": "Separation Ratio",
    "composite_recovery_score": "Global Recovery Score",
    "composite_random_baseline_mean": "Random Baseline (Composite)",
    "composite_p_vs_random": "P-value vs Random",
})

plot_df["Modality"] = plot_df["Modality"].replace({
    "trimodal_X_multivi": "Trimodal",
    "rna_X_scVI": "RNA",
    "atac_X_peakvi": "ATAC",
    "protein": "Protein"
})

plot_csv_path = output_dir / "LSC_plot_table.csv"
plot_xlsx_path = output_dir / "LSC_plot_table.xlsx"

plot_df.to_csv(plot_csv_path, index=False)
plot_df.to_excel(plot_xlsx_path, index=False)

print("\n=== CLEAN TABLE FOR FIGURES ===")
print(plot_df.round(4))
print("Saved plot table:", plot_csv_path)
print("Saved plot table:", plot_xlsx_path)


### 13. Bar plots, one per metric

In [ ]:
sns.set(style="whitegrid")

palette = {
    "Trimodal": "#000000",
    "RNA": "#1f77b4",
    "ATAC": "#ff7f0e",
    "Protein": "#2ca02c"
}


main_metrics = [
    "LSC Local Purity",
    "Neighborhood Preservation",
    "LSC Separation (Silhouette)",
    "Separation Ratio",
    "Global Recovery Score",
]

for metric in main_metrics:
    plt.figure(figsize=(6, 4))

    sns.barplot(
        data=plot_df,
        x="Modality",
        y=metric,
        hue="Modality",
        palette=palette,
        dodge=False,
        legend=False
    )

    plt.title(metric, fontsize=12)
    plt.ylabel("")
    plt.xlabel("")
    plt.tight_layout()

    save_path = output_dir / f"{metric.replace(' ', '_').replace('/', '_')}.png"

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved:", save_path)

### 14. Observed recovery versus the random baseline

In [ ]:
plot_long = plot_df.melt(
    id_vars="Modality",
    value_vars=["Global Recovery Score", "Random Baseline (Composite)"],
    var_name="Type",
    value_name="Score"
)

plt.figure(figsize=(7, 4))

sns.barplot(
    data=plot_long,
    x="Modality",
    y="Score",
    hue="Type"
)

plt.title("Observed Recovery vs Random Baseline")
plt.ylabel("Score")
plt.xlabel("")
plt.tight_layout()

baseline_fig = output_dir / "LSC_global_recovery_vs_random_baseline.png"

plt.savefig(baseline_fig, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", baseline_fig)

### 15. P-values against the random baseline

In [ ]:
plt.figure(figsize=(6, 4))

sns.barplot(
    data=plot_df,
    x="Modality",
    y="P-value vs Random",
    hue="Modality",
    palette=palette,
    dodge=False,
    legend=False
)

plt.axhline(0.05, linestyle="--", linewidth=1, color="red")
plt.title("P-values vs Random Baseline")
plt.ylabel("P-value")
plt.xlabel("")
plt.tight_layout()

p_random_fig = output_dir / "LSC_pvalues_vs_random.png"

plt.savefig(p_random_fig, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", p_random_fig)

### 17. Metric heatmap

In [ ]:
heatmap_df = plot_df.set_index("Modality")[[
    "LSC Local Purity",
    "Neighborhood Preservation",
    "LSC Separation (Silhouette)",
    "Separation Ratio",
    "Global Recovery Score",
]]

plt.figure(figsize=(8, 3.8))

sns.heatmap(
    heatmap_df,
    annot=True,
    cmap="viridis",
    cbar_kws={"label": "Score"}
)

plt.title("LSC Structure Metrics")
plt.tight_layout()

heatmap_path = output_dir / "LSC_metrics_heatmap.png"

plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", heatmap_path)